# 采购入库数据清洗确认

## 1. 读取原始数据

In [ ]:
import pandas as pd
import numpy as np
from read_data import read_raw_procurement_inbound

df = read_raw_procurement_inbound("2025-10-01", "2026-09-30")
print(f"{len(df):,} 行，{len(df.columns)} 列")
df

## 2. 已确认字段概览

购退单价仅用于补充采购单价。

In [ ]:
字段 = [
    "出入库日期", "物料编码", "物料名称", "规格", "主单位",
    "一级分类", "二级分类", "三级分类", "四级分类", "五级分类",
    "品牌", "主品牌", "货号", "供应商名称", "财务内部外供应商类型", "板块",
    "公司编码", "公司名称", "公司简称", "大区", "数据来源系统", "设备试剂分类",
    "采购类型名称", "数量", "采购单价", "采购金额(万)", "税率", "购退单价",
]
pd.DataFrame({"类型": df[字段].dtypes.astype(str), "空值": df[字段].isna().sum(),
              "空值占比": df[字段].isna().mean(), "取值数": df[字段].nunique()})

## 3. 字段验证

每项筛选异常记录到 `明细`。

### 财务内部外供应商类型

空值填“外部单位”。

In [ ]:
明细 = df.loc[df["财务内部外供应商类型"].isna(), 字段]
print(f"空值：{len(明细):,} 行")
明细

### 采购单价与购退单价

采购单价为空或为负时，用非空购退单价补充。

In [ ]:
明细 = df.loc[(df["采购单价"].isna() | df["采购单价"].lt(0)) & df["购退单价"].notna(), 字段]
print(f"可补充：{len(明细):,} 行")
明细

### 采购单价负值

购退单价非空时用于补正。

In [ ]:
明细 = df.loc[df["采购单价"].lt(0), 字段]
print(f"负单价：{len(明细):,} 行")
明细

### 采购单价与采购金额

含税单价 × 数量与含税金额的相对偏差超过 1%。

In [ ]:
偏差 = (df["采购单价"] * df["数量"] / 10000 - df["采购金额(万)"]).abs() / df["采购金额(万)"].abs().replace(0, np.nan)
明细 = df.loc[偏差.gt(0.01), 字段]
print(f"偏差超过 1%：{len(明细):,} 行；无法计算：{偏差.isna().sum():,} 行")
明细

### 采购类型名称

空值标为“未标注”。

In [ ]:
明细 = df.loc[df["采购类型名称"].isna(), 字段]
print(f"空值：{len(明细):,} 行")
明细

### 关键字段空值

物料编码、数量、金额、日期、板块或供应商名称为空。

In [ ]:
关键字段 = ["物料编码", "数量", "采购金额(万)", "出入库日期", "板块", "供应商名称"]
明细 = df.loc[df[关键字段].isna().any(axis=1), 字段]
print(f"存在空值：{len(明细):,} 行")
明细

### 文本字段格式

检查首尾空格、回车、换行和制表符。

In [ ]:
文本字段 = df[字段].select_dtypes(include=["object", "string", "str"]).columns
条件 = pd.Series(False, index=df.index)
for 列 in 文本字段:
    条件 |= df[列].astype("string").str.contains(r"[\r\n\t]|^\s|\s$", regex=True, na=False)
明细 = df.loc[条件, 字段]
print(f"存在格式问题：{len(明细):,} 行")
明细

### 公司编码与名称

同一公司编码对应多个名称。

In [ ]:
条件 = df.groupby("公司编码", dropna=False)["公司名称"].transform("nunique").gt(1)
明细 = df.loc[条件, 字段]
print(f"涉及：{len(明细):,} 行")
明细

### 数量与采购金额

退货及有数量但金额为零的记录，保留。

In [ ]:
明细 = df.loc[(df["数量"] < 0) | ((df["数量"] != 0) & (df["采购金额(万)"] == 0)), 字段]
print(f"涉及：{len(明细):,} 行")
明细

### 已确认字段相同的记录

仅为重复候选；原始整行重复待业务确认，暂不剔除。

In [ ]:
明细 = df.loc[df[字段].duplicated(keep=False), 字段]
print(f"重复候选：{len(明细):,} 行")
明细

## 4. 已确认清洗逻辑

- 文本去首尾空格及控制字符。
- 财务供应商类型空值填“外部单位”，更名为“供应商类型”。
- 采购单价为空或为负时，用非空购退单价补充。
- 采购类型名称归一为“采购模式”，空值标“未标注”。
- 出入库日期转为日期，重复行保留。

以上规则已在 `clean_inbound` 中实现。

In [ ]:
from clean.procurement_inbound import clean_inbound

cleaned, 台账 = clean_inbound(df)
print(f"行数：{len(df):,} → {len(cleaned):,}；含税金额差：{cleaned['采购金额(万)'].sum() - df['采购金额(万)'].sum():,.6f} 万元")

## 5. 清洗后完整数据

In [ ]:
cleaned